# Pipeline Sequencial — Modelo 7 → Modelo 5

## Task 1 — Contrato de integração

Este notebook implementará futuramente o pipeline sequencial formado por:

1. **Modelo 7 — CatBoost:** previsão de risco por sistema PSD, no nível de estação e plataforma;
2. **Modelo 5 — Random Forest:** previsão de risco por porta individual dentro das plataformas selecionadas.

Nesta primeira task ainda não há execução dos modelos. O objetivo é registrar o contrato que deverá ser respeitado nas próximas etapas.

## 1. Fluxo acordado

```text
Dados disponíveis até o final do dia D
                    ↓
Modelo 7 — CatBoost por plataforma
                    ↓
Plataformas com probabilidade >= 0,317380
                    ↓
Modelo 5 — Random Forest por porta
                    ↓
Portas priorizadas para inspeção
```

O Modelo 7 é executado primeiro por trabalhar no nível mais amplo e indicar que alguma porta da plataforma poderá falhar. O Modelo 5 recebe somente as portas pertencentes às plataformas alertadas e tenta localizar qual porta deve ser priorizada. A diferença de granularidade é proposital: o CatBoost ajuda o Random Forest reduzindo o espaço de busca. O score do CatBoost não será usado como feature do Random Forest nesta primeira versão, porque o Modelo 5 não foi treinado com essa variável.

## 2. Comparação das interfaces atuais

| Item | Modelo 7 — CatBoost | Modelo 5 — Random Forest | Situação |
|---|---|---|---|
| Granularidade | Plataforma × dia | Porta × dia | Compatível, complementar e proposital |
| Momento da previsão | Final de `D`, incluindo eventos de `D` | Início de `D`, usando eventos até `D-1` | Precisa alinhar |
| Horizonte | `D+1` a `D+7` | `(D, D+7]` | Mesmo tamanho; referência precisa alinhar |
| Falha positiva | ZM01 com `falha_flag == True` | Qualquer ZM01 | Precisa corrigir |
| Plataforma | `id_psd = BUT_P1` | `id_plat = BUT_1` | Precisa converter |
| Porta | Não é a unidade prevista | `id_porta = BUT_1_05` | Precisa padronizar |
| Cobertura | 11 estações e 22 plataformas | 10 estações e 480 portas | `FRE` sem detalhamento por porta |
| Features | 21 registradas em metadata | 71 mantidas por SHAP | Congelar lista do Modelo 5 |
| Limiar | `0,317380` | `0,5183` | Manter separados |
| Artefato | `.cbm` e metadata disponíveis | Modelo não serializado | Precisa exportar |

## 3. Decisões do contrato

### Data e target comuns

- O pipeline será iniciado **ao final do dia `D`**.
- O Modelo 7 manterá as features disponíveis até o final de `D`.
- O Modelo 5 manterá inicialmente sua regra original, usando o histórico até `D-1`.
- Os dois modelos observarão o horizonte futuro de `D+1` a `D+7`.
- O Modelo 7 manterá o target de falha confirmada; o Modelo 5 manterá inicialmente o target de qualquer ZM01 da porta.
- O alinhamento completo de tempo e target será um aprimoramento posterior, não um bloqueio do primeiro pipeline.

### Chaves canônicas

```text
estacao = BUT
plataforma = 1
id_psd = BUT_P1
id_porta = BUT_P1_PDM05
```

As chaves internas dos modelos serão convertidas para esse padrão na fronteira do pipeline. Uma porta só poderá chegar ao Modelo 5 quando pertencer à mesma estação e plataforma alertadas pelo Modelo 7.

### Cobertura

Enquanto o Modelo 5 não incluir `FRE`, um alerta do Modelo 7 nessa estação deverá permanecer como alerta de plataforma sem detalhamento por porta. O alerta não poderá ser descartado silenciosamente.

## 4. Resultado da Task 1

Os modelos são conceitualmente complementares e a diferença de granularidade é o mecanismo central do pipeline: o Modelo 7 encontra a plataforma em risco e o Modelo 5 procura a porta dentro dela. Para automatizar a primeira versão, será necessário:

1. salvar a lista ordenada das 71 features;
2. serializar o Random Forest e registrar seu limiar e metadata;
3. implementar a conversão das chaves;
4. tratar plataformas sem cobertura no Modelo 5;
5. preservar a regra temporal original de cada modelo durante a inferência.

Em uma iteração posterior, o Modelo 5 poderá ser retreinado com o mesmo instante e o mesmo target do Modelo 7 para permitir uma avaliação unificada de ponta a ponta. Isso não impede o primeiro teste do encadeamento.

A especificação completa está em `docs/src/content/docs/sprints/Sprint 3/pipeline-sequencial-modelos.mdx`.

# Task 6 — Configurar e executar o pipeline

A execução padrão reutiliza os modelos salvos. Ajuste o período e a pasta de saída nesta célula e execute o notebook de cima para baixo.

- `RETREINAR_MODELO_5 = False`: carrega o Random Forest existente. Ative apenas para refazer o experimento da Task 2 e substituir seu artefato.
- `AVALIAR_HISTORICO = True`: compara as previsões com o target histórico quando ele estiver disponível. Use `False` para apenas prever.
- `MAXIMO_PORTAS_RECOMENDADAS_POR_PLATAFORMA = 1`: limita a lista operacional às portas de maior risco de cada plataforma e data. A saída completa continua preservada.
- `PASTA_SAIDA`: diretório dentro de `models/pipeline-sequencial/`. Os modelos salvos permanecem em `artefatos/`.

Ao final, `status_plataformas_pipeline.csv` registra também plataformas sem cobertura ou sem portas acima do limiar. O relatório `resumo_execucao_pipeline.json` resume a execução.


In [ ]:
RETREINAR_MODELO_5 = False
TREINAR_MODELO_5_COM_SCORE_7 = True
USAR_MODELO_5_COM_SCORE_7 = True
META_RECALL_EXPERIMENTO = 0.75
AVALIAR_HISTORICO = True
INICIO_INFERENCIA_MODELO_7 = '2025-10-01'
FIM_INFERENCIA_MODELO_7 = '2026-03-21'
MAXIMO_PORTAS_RECOMENDADAS_POR_PLATAFORMA = 1
PASTA_SAIDA = 'artefatos'


# Task 2 — Preparar o Modelo 5 para inferência

Esta etapa reconstrói a configuração selecionada do Random Forest sem modificar os arquivos das sprints anteriores. O painel e as ordens antigas são somente entradas de leitura. Como o objeto treinado na Sprint 2 não foi serializado, essa reconstrução gerou o artefato atual. Agora o treino e a exportação abaixo só são executados quando `RETREINAR_MODELO_5 = True`; por padrão, o artefato existente é reutilizado.

Os novos artefatos serão gravados exclusivamente em `models/pipeline-sequencial/artefatos/`:

- `modelo_5_random_forest.joblib`;
- `metadata_modelo_5.json`.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json
import re

import joblib
import numpy as np
import pandas as pd
import sklearn
from IPython.display import display
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_score,
    precision_recall_curve,
    recall_score,
    roc_auc_score,
)


def localizar_raiz_repositorio():
    caminho_atual = Path.cwd().resolve()
    for candidato in (caminho_atual, *caminho_atual.parents):
        if (candidato / 'models/modelo-5/painel.parquet').exists():
            return candidato
    raise FileNotFoundError('Não foi possível localizar a raiz do repositório g04.')


RAIZ_REPOSITORIO = localizar_raiz_repositorio()
DIRETORIO_MODELO_5 = RAIZ_REPOSITORIO / 'models/modelo-5'
DIRETORIO_PIPELINE = RAIZ_REPOSITORIO / 'models/pipeline-sequencial'
DIRETORIO_MODELOS_PIPELINE = DIRETORIO_PIPELINE / 'artefatos'
DIRETORIO_ARTEFATOS = (DIRETORIO_PIPELINE / PASTA_SAIDA).resolve()
if not DIRETORIO_ARTEFATOS.is_relative_to(DIRETORIO_PIPELINE.resolve()):
    raise ValueError('PASTA_SAIDA deve ficar dentro de models/pipeline-sequencial.')
DIRETORIO_ARTEFATOS.mkdir(parents=True, exist_ok=True)
DIRETORIO_PLANILHAS = DIRETORIO_ARTEFATOS / 'planilhas'
DIRETORIO_PLANILHAS.mkdir(parents=True, exist_ok=True)

caminho_painel = DIRETORIO_MODELO_5 / 'painel.parquet'
caminho_zm01 = RAIZ_REPOSITORIO / 'data/Corretiva Emergencial/processados/ZM01_limpa.xlsx'

if not RETREINAR_MODELO_5:
    artefatos_inferencia = (
        ('modelo_5_com_score_modelo_7.joblib', 'metadata_modelo_5_com_score_modelo_7.json')
        if USAR_MODELO_5_COM_SCORE_7
        else ('modelo_5_random_forest.joblib', 'metadata_modelo_5.json')
    )
    for nome in artefatos_inferencia:
        if not (DIRETORIO_MODELOS_PIPELINE / nome).is_file():
            raise FileNotFoundError(
                f'Artefato ausente: {nome}. Execute a Task 2 com '
                'RETREINAR_MODELO_5 = True para reconstruí-lo.'
            )

painel = pd.read_parquet(caminho_painel)
print(f'Painel carregado: {painel.shape[0]:,} linhas × {painel.shape[1]} colunas')
print(f'Período: {painel["Data"].min().date()} a {painel["Data"].max().date()}')
print(f'Portas: {painel["id_porta"].nunique()}')

In [ ]:
# Reproduz somente as features criadas dentro do notebook original do Modelo 5.
# Nenhum arquivo antigo é alterado.
PAT_LOC = re.compile(
    r'^SP04-6-PSD-([A-Z]{3})(?:-PLT(\d)|-P(\d)S\d)?(?:-PP(\d{2})(PDM|PFX|PSE))?'
)


def carregar_zm01_porta(caminho):
    ordens = pd.read_excel(caminho, sheet_name='ZM01_limpa')
    correspondencias = ordens['local_de_instalacao'].apply(
        lambda valor: PAT_LOC.match(valor) if isinstance(valor, str) else None
    )
    ordens['est'] = [item.group(1) if item else None for item in correspondencias]
    ordens['via'] = [
        (item.group(2) or item.group(3)) if item else None
        for item in correspondencias
    ]
    ordens['porta'] = [
        int(item.group(4)) if item and item.group(4) else None
        for item in correspondencias
    ]
    ordens = ordens.dropna(
        subset=['est', 'via', 'porta', 'inicio_avaria_datetime']
    ).copy()
    ordens['porta'] = ordens['porta'].astype(int)
    ordens['id_porta'] = (
        ordens['est'] + '_' + ordens['via'] + '_'
        + ordens['porta'].astype(str).str.zfill(2)
    )
    ordens['Data'] = pd.to_datetime(ordens['inicio_avaria_datetime']).dt.normalize()
    return ordens


zm01 = carregar_zm01_porta(caminho_zm01)
eventos = (
    zm01[['id_porta', 'Data']]
    .dropna()
    .drop_duplicates()
    .sort_values(['id_porta', 'Data'])
    .reset_index(drop=True)
)
eventos['ordem'] = eventos.groupby('id_porta').cumcount() + 1
eventos['diff'] = eventos.groupby('id_porta')['Data'].diff().dt.days
eventos['intervalo_medio_ate_aqui'] = (
    eventos.groupby('id_porta')['diff']
    .apply(lambda serie: serie.expanding().mean())
    .reset_index(level=0, drop=True)
)

referencias = painel[['id_porta', 'Data']].copy()
referencias['_idx'] = painel.index
historico = pd.merge_asof(
    referencias.sort_values('Data'),
    eventos[['id_porta', 'Data', 'ordem', 'intervalo_medio_ate_aqui']].sort_values('Data'),
    on='Data',
    by='id_porta',
    direction='backward',
    allow_exact_matches=False,
)

painel['n_corretivas_historico_total'] = (
    pd.Series(historico['ordem'].values, index=historico['_idx'].values)
    .reindex(painel.index)
    .fillna(0)
)
intervalos = pd.Series(
    historico['intervalo_medio_ate_aqui'].values,
    index=historico['_idx'].values,
).reindex(painel.index)
painel['intervalo_medio_zm01'] = intervalos.fillna(intervalos.median())
painel['razao_atraso_zm01'] = painel['dias_desde_zm01'] / painel['intervalo_medio_zm01']
painel['dias_desde_ultima_corretiva'] = painel[['dias_desde_zm01', 'dias_desde_zm02']].min(axis=1)

painel['rampa_7d'] = painel['n_total_7d'] - (painel['n_total_14d'] - painel['n_total_7d'])
painel['razao_7d_30d'] = painel['n_total_7d'] / (painel['n_total_30d'] / 4 + 1)
painel['frac_tecnicos_30d'] = painel['n_tecnicos_30d'] / (painel['n_total_30d'] + 1)
painel['est_cod'] = painel['est'].astype('category').cat.codes
painel['via_cod'] = painel['via'].astype(int)

print('Features adicionais do notebook original reproduzidas.')

In [ ]:
COLUNAS_EXCLUIDAS = {
    'id_porta', 'Data', 'est', 'via', 'id_plat', 'y',
    'n_falhas_7d_futuro', 'cobertura_30d', 'mes',
}
FEATURES_REMOVIDAS_SHAP = [
    'n_CSD_3d', 'n_FO_3d', 'zm02_30d', 'n_RCE_7d',
    'n_FO_1d', 'n_OA_1d', 'dia_semana', 'n_RCE_3d',
    'n_CSD_1d', 'n_CA_1d', 'fim_de_semana', 'n_RCE_1d',
]

features_candidatas = [
    coluna for coluna in painel.columns if coluna not in COLUNAS_EXCLUIDAS
]
features_modelo_5 = [
    coluna for coluna in features_candidatas
    if coluna not in FEATURES_REMOVIDAS_SHAP
]

assert len(features_candidatas) == 83, (
    f'Esperadas 83 candidatas, encontradas {len(features_candidatas)}.'
)
assert len(features_modelo_5) == 71, (
    f'Esperadas 71 features finais, encontradas {len(features_modelo_5)}.'
)
assert not painel[features_modelo_5].isna().any().any()
assert np.isfinite(painel[features_modelo_5].to_numpy()).all()

print(f'Features candidatas: {len(features_candidatas)}')
print(f'Features removidas: {len(FEATURES_REMOVIDAS_SHAP)}')
print(f'Features finais congeladas: {len(features_modelo_5)}')
print(features_modelo_5)

In [ ]:
if RETREINAR_MODELO_5:
    FIM_TREINO = pd.Timestamp('2025-04-23')
    INICIO_TESTE = pd.Timestamp('2025-05-01')
    INICIO_VALIDACAO = pd.Timestamp('2025-01-31')

    treino = painel[painel['Data'] <= FIM_TREINO].copy()
    teste = painel[painel['Data'] >= INICIO_TESTE].copy()
    treino_fit = treino[
        treino['Data'] <= INICIO_VALIDACAO - pd.Timedelta(days=8)
    ].copy()
    validacao = treino[treino['Data'] >= INICIO_VALIDACAO].copy()


    def selecionar_limiar(y_real, probabilidades):
        candidatos = np.unique(
            np.quantile(probabilidades, np.linspace(0.50, 0.999, 200))
        )
        return max(
            (
                f1_score(
                    y_real,
                    (probabilidades >= limiar).astype(int),
                    zero_division=0,
                ),
                limiar,
            )
            for limiar in candidatos
        )[1]


    parametros_modelo_5 = {
        'n_estimators': 400,
        'max_depth': 14,
        'min_samples_leaf': 40,
        'class_weight': 'balanced',
        'n_jobs': -1,
        'random_state': 42,
    }

    modelo_validacao = RandomForestClassifier(**parametros_modelo_5)
    modelo_validacao.fit(
        treino_fit[features_modelo_5],
        treino_fit['y'],
    )
    probabilidades_validacao = modelo_validacao.predict_proba(
        validacao[features_modelo_5]
    )[:, 1]
    limiar_modelo_5 = selecionar_limiar(
        validacao['y'], probabilidades_validacao
    )

    modelo_5 = RandomForestClassifier(**parametros_modelo_5)
    modelo_5.fit(treino[features_modelo_5], treino['y'])
    probabilidades_teste = modelo_5.predict_proba(teste[features_modelo_5])[:, 1]
    predicoes_teste = (probabilidades_teste >= limiar_modelo_5).astype(int)
    tn, fp, fn, tp = confusion_matrix(teste['y'], predicoes_teste).ravel()
    metricas_teste = {
        'precision': float(precision_score(teste['y'], predicoes_teste, zero_division=0)),
        'recall': float(recall_score(teste['y'], predicoes_teste, zero_division=0)),
        'f1': float(f1_score(teste['y'], predicoes_teste, zero_division=0)),
        'pr_auc': float(average_precision_score(teste['y'], probabilidades_teste)),
        'roc_auc': float(roc_auc_score(teste['y'], probabilidades_teste)),
        'alertas': int(predicoes_teste.sum()),
        'tn': int(tn),
        'fp': int(fp),
        'fn': int(fn),
        'tp': int(tp),
    }

    print(f'Limiar selecionado: {limiar_modelo_5:.6f}')
    print(json.dumps(metricas_teste, indent=2, ensure_ascii=False))
else:
    print('Treinamento ignorado: o pipeline usará o Random Forest salvo.')


In [ ]:
if RETREINAR_MODELO_5:
    caminho_modelo_5 = DIRETORIO_MODELOS_PIPELINE / 'modelo_5_random_forest.joblib'
    caminho_metadata_5 = DIRETORIO_MODELOS_PIPELINE / 'metadata_modelo_5.json'

    metadata_modelo_5 = {
        'modelo': 'Modelo 5 — Random Forest',
        'algoritmo': 'RandomForestClassifier',
        'arquivo_modelo': caminho_modelo_5.name,
        'fonte_original': 'models/modelo-5/modelo_5.ipynb',
        'unidade_analise': 'porta_dia',
        'chave_interna': 'id_porta no formato EST_VIA_PORTA',
        'momento_previsao': 'inicio do dia D',
        'dados_permitidos': 'eventos anteriores ao dia D',
        'horizonte': '(D, D+7]',
        'target': 'y',
        'definicao_target': 'existe qualquer ZM01 da mesma porta em (D, D+7]',
        'limiar': float(limiar_modelo_5),
        'quantidade_features': len(features_modelo_5),
        'features': features_modelo_5,
        'tipos_features': {
            coluna: str(painel[coluna].dtype)
            for coluna in features_modelo_5
        },
        'criterio_features': (
            'mesmo conjunto de 71 features mantidas após a seleção SHAP; '
            'ordem determinística herdada do painel'
        ),
        'features_removidas_shap': FEATURES_REMOVIDAS_SHAP,
        'hiperparametros': parametros_modelo_5,
        'periodos': {
            'treino_inicio': str(treino['Data'].min().date()),
            'treino_fim': str(treino['Data'].max().date()),
            'validacao_inicio': str(validacao['Data'].min().date()),
            'validacao_fim': str(validacao['Data'].max().date()),
            'teste_inicio': str(teste['Data'].min().date()),
            'teste_fim': str(teste['Data'].max().date()),
        },
        'metricas_teste': metricas_teste,
        'bibliotecas': {
            'python': __import__('sys').version.split()[0],
            'pandas': pd.__version__,
            'numpy': np.__version__,
            'scikit_learn': sklearn.__version__,
            'joblib': joblib.__version__,
        },
        'gerado_em_utc': datetime.now(timezone.utc).isoformat(),
    }

    joblib.dump(modelo_5, caminho_modelo_5, compress=3)
    caminho_metadata_5.write_text(
        json.dumps(metadata_modelo_5, indent=2, ensure_ascii=False) + '\n',
        encoding='utf-8',
    )

    modelo_recarregado = joblib.load(caminho_modelo_5)
    assert list(modelo_recarregado.feature_names_in_) == features_modelo_5
    probabilidades_recarregadas = modelo_recarregado.predict_proba(
        teste[features_modelo_5]
    )[:, 1]
    assert np.allclose(probabilidades_recarregadas, probabilidades_teste)

    print(f'Modelo salvo e validado em: {caminho_modelo_5}')
    print(f'Metadata salvo em: {caminho_metadata_5}')
else:
    print('Artefato e metadata do Modelo 5 preservados para inferência.')


# Task 5 — Gerar as previsões do CatBoost no pipeline

Este bloco é executado antes da Task 3 porque o encaminhamento usa as previsões calculadas aqui. Ele carrega o CatBoost treinado e reconstrói suas 21 features a partir da tabela mestra diária.

O período padrão preserva o teste histórico (01/10/2025 a 21/03/2026). Para outra data, ajuste as duas datas na configuração inicial. É necessário fornecer o histórico diário, com pelo menos 30 dias; a recência das manutenções usa todo o histórico disponível. A previsão usa dados até o final de D e mantém o horizonte de sete dias, sem precisar do target ou de registros futuros.

Se faltar a dependência no kernel, execute `%pip install catboost==1.2.10` em uma célula e reinicie o kernel.


In [ ]:
try:
    import catboost
    from catboost import CatBoostClassifier, Pool
except ImportError as erro:
    raise ImportError(
        'Instale %pip install catboost==1.2.10 no kernel e reinicie-o.'
    ) from erro

from hashlib import sha256

DIRETORIO_MODELO_7 = RAIZ_REPOSITORIO / 'models/modelo-7'
CAMINHO_MESTRA_MODELO_7 = (
    RAIZ_REPOSITORIO / 'data/processados/modelo-7/tabela_mestra_psd.csv'
)
CAMINHO_METADATA_MODELO_7 = DIRETORIO_MODELO_7 / 'metadata_modelo.json'
CAMINHO_CATBOOST = DIRETORIO_MODELO_7 / 'modelo_catboost.cbm'
INICIO_INFERENCIA_MODELO_7 = pd.Timestamp(INICIO_INFERENCIA_MODELO_7)
FIM_INFERENCIA_MODELO_7 = pd.Timestamp(FIM_INFERENCIA_MODELO_7)

with CAMINHO_METADATA_MODELO_7.open(encoding='utf-8') as arquivo:
    metadata_modelo_7 = json.load(arquivo)

COLUNAS_DIARIAS_MODELO_7 = [
    'qtd_anomalias_criticas_30s', 'qtd_criticas_pico_manha',
    'qtd_criticas_pico_tarde', 'duracao_media_alarmes_seg',
    'duracao_maxima_alarme_seg', 'qtd_portas_com_alarme',
    'qtd_corretivas_emergenciais', 'qtd_corretivas_programadas',
    'qtd_preventivas_nao_canceladas',
]


def preparar_entrada_modelo_7(tabela_diaria, metadata, inicio, fim):
    """Reproduz as features do notebook 02 com informações disponíveis até D."""
    inicio, fim = pd.Timestamp(inicio), pd.Timestamp(fim)
    if (
        pd.isna(inicio) or pd.isna(fim) or inicio > fim
        or inicio.tzinfo is not None or fim.tzinfo is not None
        or inicio != inicio.normalize() or fim != fim.normalize()
    ):
        raise ValueError('Informe datas diárias válidas, sem fuso, com início <= fim.')

    obrigatorias = [
        'data_referencia', 'id_psd', 'estacao_padrao', 'plataforma_padrao',
        *COLUNAS_DIARIAS_MODELO_7,
    ]
    faltantes = sorted(set(obrigatorias) - set(tabela_diaria.columns))
    if faltantes:
        raise ValueError(f'Colunas ausentes na tabela mestra: {faltantes}')
    # Seleção explícita: nenhuma coluna de target participa da preparação.
    historico = tabela_diaria[obrigatorias].copy()
    historico['data_referencia'] = pd.to_datetime(
        historico['data_referencia'], errors='raise'
    )
    datas = historico['data_referencia']
    if (
        datas.isna().any() or datas.dt.tz is not None
        or not datas.eq(datas.dt.normalize()).all()
    ):
        raise ValueError('A tabela mestra deve usar datas diárias válidas, sem fuso.')
    historico = historico.loc[datas <= fim].copy()
    if historico.empty or historico[obrigatorias].isna().any().any():
        raise ValueError('Histórico vazio ou valores ausentes nas entradas diárias.')

    historico['plataforma_padrao'] = pd.to_numeric(
        historico['plataforma_padrao'], errors='raise'
    )
    if not historico['plataforma_padrao'].isin([1, 2]).all():
        raise ValueError('Plataforma deve ser 1 ou 2.')
    historico['plataforma_padrao'] = historico['plataforma_padrao'].astype(int)
    estacao = historico['estacao_padrao'].astype(str)
    id_esperado = estacao + '_P' + historico['plataforma_padrao'].astype(str)
    if (
        not estacao.str.fullmatch(r'[A-Z]{3}').all()
        or not historico['id_psd'].eq(id_esperado).all()
    ):
        raise ValueError('id_psd não corresponde à estação e à plataforma.')
    if historico.duplicated(['id_psd', 'data_referencia']).any():
        raise ValueError('A tabela contém plataforma e data duplicadas.')
    historico[COLUNAS_DIARIAS_MODELO_7] = historico[
        COLUNAS_DIARIAS_MODELO_7
    ].apply(pd.to_numeric, errors='raise')
    numeros = historico[COLUNAS_DIARIAS_MODELO_7].to_numpy(dtype=float)
    if not np.isfinite(numeros).all() or (numeros < 0).any():
        raise ValueError('Contagens e durações devem ser finitas e não negativas.')

    historico = historico.sort_values(
        ['id_psd', 'data_referencia']
    ).reset_index(drop=True)
    intervalos = historico.groupby('id_psd')['data_referencia'].diff().dropna()
    if not intervalos.eq(pd.Timedelta(days=1)).all():
        raise ValueError('Há dias ausentes na tabela mestra; confira a cobertura.')

    nomes_dias = {
        0: 'segunda', 1: 'terca', 2: 'quarta', 3: 'quinta',
        4: 'sexta', 5: 'sabado', 6: 'domingo',
    }
    dias = historico['data_referencia'].dt.dayofweek
    historico['dia_semana'] = dias.map(nomes_dias)
    historico['domingo_flag'] = dias.eq(6).astype(int)
    historico['qtd_criticas_pico_dia'] = (
        historico['qtd_criticas_pico_manha'] + historico['qtd_criticas_pico_tarde']
    )
    historico['dia_com_anomalia_critica'] = (
        historico['qtd_anomalias_criticas_30s'] > 0
    ).astype(int)
    grupo = historico.groupby('id_psd', sort=False)
    for origem, destino in {
        'qtd_anomalias_criticas_30s': 'qtd_anomalias_criticas_30s_7d',
        'qtd_criticas_pico_dia': 'qtd_criticas_pico_7d',
        'dia_com_anomalia_critica': 'dias_com_anomalia_critica_7d',
    }.items():
        historico[destino] = grupo[origem].transform(
            lambda serie: serie.rolling(7, min_periods=7).sum()
        )
    historico['duracao_maxima_alarme_7d'] = grupo[
        'duracao_maxima_alarme_seg'
    ].transform(lambda serie: serie.rolling(7, min_periods=7).max())
    denominador = historico['qtd_anomalias_criticas_30s_7d']
    historico['proporcao_criticas_pico_7d'] = (
        historico['qtd_criticas_pico_7d'] / denominador.replace(0, np.nan)
    )
    historico.loc[denominador.eq(0), 'proporcao_criticas_pico_7d'] = 0.0

    for origem, destino in {
        'qtd_corretivas_emergenciais': 'dias_desde_ultima_corretiva',
        'qtd_preventivas_nao_canceladas': 'dias_desde_ultima_preventiva',
    }.items():
        datas_evento = historico['data_referencia'].where(historico[origem] > 0)
        ultima_data = datas_evento.groupby(historico['id_psd']).ffill()
        historico[destino] = (historico['data_referencia'] - ultima_data).dt.days

    for origem, destino in {
        'qtd_corretivas_emergenciais': 'qtd_corretivas_emergenciais_30d',
        'qtd_corretivas_programadas': 'qtd_corretivas_programadas_30d',
        'qtd_preventivas_nao_canceladas': 'qtd_preventivas_nao_canceladas_30d',
    }.items():
        historico[destino] = grupo[origem].transform(
            lambda serie: serie.rolling(30, min_periods=30).sum()
        )
    categoricas = metadata['features_categoricas']
    numericas = metadata['features_numericas']
    entrada = historico.loc[
        historico['data_referencia'].between(inicio, fim),
        ['data_referencia', *categoricas, *numericas],
    ].copy()
    dias_pedidos = (fim - inicio).days + 1
    if len(entrada) != historico['id_psd'].nunique() * dias_pedidos:
        raise ValueError('O período solicitado não está coberto por todas as plataformas.')
    # Sem manutenção anterior: NaN, como no treinamento original do CatBoost.
    recencias = {'dias_desde_ultima_corretiva', 'dias_desde_ultima_preventiva'}
    if entrada[[c for c in numericas if c not in recencias]].isna().any().any():
        raise ValueError('São necessários pelo menos 30 dias completos de histórico.')
    for coluna in categoricas:
        entrada[coluna] = entrada[coluna].astype(str)
    entrada[numericas] = entrada[numericas].astype(float)
    if np.isinf(entrada[numericas].to_numpy()).any():
        raise ValueError('As features contêm valores infinitos.')
    return entrada.sort_values(['data_referencia', 'id_psd']).reset_index(drop=True)


tabela_diaria_modelo_7 = pd.read_csv(
    CAMINHO_MESTRA_MODELO_7, parse_dates=['data_referencia']
)
entrada_modelo_7 = preparar_entrada_modelo_7(
    tabela_diaria_modelo_7, metadata_modelo_7,
    INICIO_INFERENCIA_MODELO_7, FIM_INFERENCIA_MODELO_7,
)
print(f'Entradas do CatBoost: {len(entrada_modelo_7):,} linhas, 21 features.')


In [ ]:
def prever_plataformas_modelo_7(entrada, modelo, metadata):
    categoricas = metadata['features_categoricas']
    features = categoricas + metadata['features_numericas']
    if list(modelo.feature_names_) != features:
        raise ValueError('Ordem das features do CatBoost difere do metadata.')
    if modelo.get_cat_feature_indices() != [features.index(c) for c in categoricas]:
        raise ValueError('As features categóricas diferem do modelo treinado.')
    if list(modelo.classes_) != [0, 1]:
        raise ValueError('Classes inesperadas no CatBoost.')
    limiar = float(metadata['limiar'])
    if not np.isfinite(limiar) or not 0 <= limiar <= 1:
        raise ValueError('Limiar inválido no metadata do Modelo 7.')
    pool = Pool(entrada[features], cat_features=categoricas)
    previsoes = entrada[['data_referencia', 'id_psd']].copy()
    previsoes['probabilidade_falha_7d'] = modelo.predict_proba(pool)[:, 1]
    previsoes['predicao_falha_7d'] = (
        previsoes['probabilidade_falha_7d'] >= limiar
    ).astype(int)
    return previsoes


modelo_7_inferencia = CatBoostClassifier()
modelo_7_inferencia.load_model(str(CAMINHO_CATBOOST))
previsoes_modelo_7 = prever_plataformas_modelo_7(
    entrada_modelo_7, modelo_7_inferencia, metadata_modelo_7
)
caminho_previsoes_geradas_modelo_7 = (
    DIRETORIO_PLANILHAS / 'previsoes_modelo_7_pipeline.csv'
)
previsoes_modelo_7.to_csv(
    caminho_previsoes_geradas_modelo_7, index=False, date_format='%Y-%m-%d',
    float_format='%.15g',
)
registro_execucao_modelo_7 = {
    'modo': 'inferencia_catboost_a_partir_da_tabela_mestra',
    'fonte': str(CAMINHO_MESTRA_MODELO_7.relative_to(RAIZ_REPOSITORIO)),
    'sha256_tabela_mestra': sha256(CAMINHO_MESTRA_MODELO_7.read_bytes()).hexdigest(),
    'modelo': str(CAMINHO_CATBOOST.relative_to(RAIZ_REPOSITORIO)),
    'sha256_modelo': sha256(CAMINHO_CATBOOST.read_bytes()).hexdigest(),
    'sha256_metadata': sha256(CAMINHO_METADATA_MODELO_7.read_bytes()).hexdigest(),
    'catboost': catboost.__version__,
    'features': metadata_modelo_7['features_categoricas']
        + metadata_modelo_7['features_numericas'],
    'limiar': float(metadata_modelo_7['limiar']),
    'horizonte_dias': 7,
    'periodo': {
        'inicio': str(INICIO_INFERENCIA_MODELO_7.date()),
        'fim': str(FIM_INFERENCIA_MODELO_7.date()),
    },
    'linhas_plataforma_dia': len(previsoes_modelo_7),
    'plataformas': int(previsoes_modelo_7['id_psd'].nunique()),
    'alertas_plataforma_dia': int(previsoes_modelo_7['predicao_falha_7d'].sum()),
    'arquivo_previsoes': caminho_previsoes_geradas_modelo_7.name,
    'gerado_em_utc': datetime.now(timezone.utc).isoformat(),
}
(DIRETORIO_ARTEFATOS / 'metadata_execucao_modelo_7.json').write_text(
    json.dumps(registro_execucao_modelo_7, indent=2, ensure_ascii=False) + '\n',
    encoding='utf-8',
)
print(f'CatBoost executado: {len(previsoes_modelo_7):,} previsões de plataforma/dia.')
print(f"Alertas de plataforma/dia: {registro_execucao_modelo_7['alertas_plataforma_dia']:,}")
display(previsoes_modelo_7.head())


# Task 3 — Padronizar chaves e preparar o encaminhamento

Esta etapa implementa a ponte entre as duas granularidades. As chaves internas dos modelos são convertidas para o padrão do pipeline, e cada plataforma alertada pelo Modelo 7 é associada às portas correspondentes do Modelo 5.

O encaminhamento ocorre apenas quando existem tanto o mapeamento da plataforma quanto as features do Modelo 5 para a data de referência. Os demais casos recebem um status explícito e não são descartados silenciosamente.

In [ ]:
DIRETORIO_MODELO_7 = RAIZ_REPOSITORIO / 'models/modelo-7'
caminho_metadata_modelo_7 = DIRETORIO_MODELO_7 / 'metadata_modelo.json'

PADRAO_PSD = re.compile(r'^(?P<estacao>[A-Z]{3})_P(?P<plataforma>\d+)$')
PADRAO_PLATAFORMA_MODELO_5 = re.compile(
    r'^(?P<estacao>[A-Z]{3})_(?P<plataforma>\d+)$'
)
PADRAO_PORTA_MODELO_5 = re.compile(
    r'^(?P<estacao>[A-Z]{3})_(?P<plataforma>\d+)_(?P<porta>\d{2})$'
)


def validar_id_psd(valor):
    if not isinstance(valor, str) or PADRAO_PSD.fullmatch(valor) is None:
        raise ValueError(f'Chave de PSD inválida: {valor!r}')
    return valor


def plataforma_modelo_5_para_pipeline(valor):
    correspondencia = PADRAO_PLATAFORMA_MODELO_5.fullmatch(str(valor))
    if correspondencia is None:
        raise ValueError(f'Chave de plataforma inválida no Modelo 5: {valor!r}')
    return (
        f"{correspondencia.group('estacao')}_P"
        f"{int(correspondencia.group('plataforma'))}"
    )


def porta_modelo_5_para_pipeline(valor):
    correspondencia = PADRAO_PORTA_MODELO_5.fullmatch(str(valor))
    if correspondencia is None:
        raise ValueError(f'Chave de porta inválida no Modelo 5: {valor!r}')
    return (
        f"{correspondencia.group('estacao')}_P"
        f"{int(correspondencia.group('plataforma'))}_PDM"
        f"{int(correspondencia.group('porta')):02d}"
    )


mapa_portas = (
    painel[['id_plat', 'id_porta', 'est', 'via', 'num_porta']]
    .drop_duplicates()
    .rename(columns={
        'id_plat': 'id_plat_modelo_5',
        'id_porta': 'id_porta_modelo_5',
        'est': 'estacao',
        'via': 'plataforma',
    })
)
mapa_portas['plataforma'] = mapa_portas['plataforma'].astype(int)
mapa_portas['id_psd'] = mapa_portas['id_plat_modelo_5'].map(
    plataforma_modelo_5_para_pipeline
)
mapa_portas['id_porta_pipeline'] = mapa_portas['id_porta_modelo_5'].map(
    porta_modelo_5_para_pipeline
)
mapa_portas = mapa_portas.sort_values(
    ['id_psd', 'num_porta']
).reset_index(drop=True)

assert len(mapa_portas) == 480
assert mapa_portas['id_psd'].nunique() == 20
assert not mapa_portas['id_porta_modelo_5'].duplicated().any()
assert not mapa_portas['id_porta_pipeline'].duplicated().any()
assert mapa_portas.groupby('id_psd').size().eq(24).all()

print(f'Portas mapeadas: {len(mapa_portas)}')
print(f'Plataformas cobertas pelo Modelo 5: {mapa_portas["id_psd"].nunique()}')
display(mapa_portas.head())

In [ ]:
with caminho_metadata_modelo_7.open(encoding='utf-8') as arquivo:
    metadata_modelo_7 = json.load(arquivo)

# A Task 5 calculou previsoes_modelo_7 diretamente com o CatBoost.
# O encaminhamento consome o arquivo recém-gerado pelo pipeline.
previsoes_modelo_7 = pd.read_csv(
    DIRETORIO_PLANILHAS / 'previsoes_modelo_7_pipeline.csv',
    parse_dates=['data_referencia'],
)
colunas_esperadas_modelo_7 = {
    'data_referencia',
    'id_psd',
    'probabilidade_falha_7d',
    'predicao_falha_7d',
}
assert colunas_esperadas_modelo_7.issubset(previsoes_modelo_7.columns)
assert not previsoes_modelo_7.duplicated(
    ['data_referencia', 'id_psd']
).any()
previsoes_modelo_7['id_psd'].map(validar_id_psd)

limiar_modelo_7 = float(metadata_modelo_7['limiar'])
saida_modelo_7 = previsoes_modelo_7[[
    'data_referencia',
    'id_psd',
    'probabilidade_falha_7d',
]].rename(columns={
    'probabilidade_falha_7d': 'probabilidade_plataforma',
})
saida_modelo_7['limiar_modelo_7'] = limiar_modelo_7
saida_modelo_7['alerta_plataforma'] = (
    saida_modelo_7['probabilidade_plataforma'] >= limiar_modelo_7
).astype(int)
assert np.array_equal(
    saida_modelo_7['alerta_plataforma'].to_numpy(),
    previsoes_modelo_7['predicao_falha_7d'].to_numpy(),
)

plataformas_modelo_5 = set(mapa_portas['id_psd'])
inicio_painel_modelo_5 = painel['Data'].min()
fim_painel_modelo_5 = painel['Data'].max()
portas_disponiveis_data = (
    painel[['Data', 'id_porta']]
    .drop_duplicates()
    .rename(columns={
        'Data': 'data_referencia',
        'id_porta': 'id_porta_modelo_5',
    })
    .merge(
        mapa_portas,
        on='id_porta_modelo_5',
        how='inner',
        validate='many_to_one',
    )
)
quantidade_portas_disponiveis = (
    portas_disponiveis_data.groupby(
        ['data_referencia', 'id_psd']
    )['id_porta_modelo_5']
    .nunique()
    .rename('quantidade_portas_com_features_modelo_5')
    .reset_index()
)
saida_modelo_7['cobertura_plataforma_modelo_5'] = (
    saida_modelo_7['id_psd'].isin(plataformas_modelo_5)
)
saida_modelo_7['cobertura_temporal_modelo_5'] = (
    saida_modelo_7['data_referencia'].between(
        inicio_painel_modelo_5,
        fim_painel_modelo_5,
    )
)
saida_modelo_7 = saida_modelo_7.merge(
    quantidade_portas_disponiveis,
    on=['data_referencia', 'id_psd'],
    how='left',
    validate='one_to_one',
)
saida_modelo_7['quantidade_portas_com_features_modelo_5'] = (
    saida_modelo_7['quantidade_portas_com_features_modelo_5']
    .fillna(0)
    .astype(int)
)
saida_modelo_7['cobertura_features_modelo_5'] = (
    saida_modelo_7['quantidade_portas_com_features_modelo_5'].gt(0)
)
plataforma_deve_ser_avaliada = (
    pd.Series(True, index=saida_modelo_7.index)
    if USAR_MODELO_5_COM_SCORE_7
    else saida_modelo_7['alerta_plataforma'].eq(1)
)
saida_modelo_7['status_encaminhamento'] = np.select(
    [
        ~plataforma_deve_ser_avaliada,
        ~saida_modelo_7['cobertura_plataforma_modelo_5'],
        ~saida_modelo_7['cobertura_features_modelo_5'],
        saida_modelo_7['quantidade_portas_com_features_modelo_5'].lt(24),
    ],
    [
        'sem_alerta_modelo_7',
        'alerta_sem_mapeamento_portas',
        'alerta_sem_features_modelo_5',
        'encaminhar_modelo_5_cobertura_parcial',
    ],
    default='encaminhar_modelo_5',
)

plataformas_elegiveis = saida_modelo_7[
    saida_modelo_7['status_encaminhamento'].str.startswith(
        'encaminhar_modelo_5'
    )
].copy()
portas_encaminhadas = plataformas_elegiveis.merge(
    portas_disponiveis_data,
    on=['data_referencia', 'id_psd'],
    how='inner',
    validate='one_to_many',
)
assert len(portas_encaminhadas) == int(
    plataformas_elegiveis[
        'quantidade_portas_com_features_modelo_5'
    ].sum()
)
assert not portas_encaminhadas.duplicated(
    ['data_referencia', 'id_porta_pipeline']
).any()

resumo_status = (
    saida_modelo_7['status_encaminhamento']
    .value_counts()
    .rename_axis('status')
    .to_frame('quantidade')
)
plataformas_sem_mapeamento = sorted(
    set(saida_modelo_7['id_psd']) - plataformas_modelo_5
)

display(resumo_status)
print(f'Plataformas sem portas mapeadas: {plataformas_sem_mapeamento}')
print(f'Pares plataforma-data encaminhados: {len(plataformas_elegiveis):,}')
print(f'Linhas de porta encaminhadas: {len(portas_encaminhadas):,}')

In [ ]:
caminho_mapa_portas = DIRETORIO_PLANILHAS / 'mapa_chaves_portas.csv'
caminho_saida_modelo_7 = (
    DIRETORIO_PLANILHAS / 'saida_intermediaria_modelo_7.csv'
)
caminho_portas_encaminhadas = (
    DIRETORIO_PLANILHAS / 'portas_encaminhadas_modelo_5.csv'
)
caminho_metadata_integracao = (
    DIRETORIO_ARTEFATOS / 'metadata_integracao_chaves.json'
)

mapa_portas.to_csv(caminho_mapa_portas, index=False)
saida_modelo_7.to_csv(
    caminho_saida_modelo_7,
    index=False,
    date_format='%Y-%m-%d',
    float_format='%.15g',
)
portas_encaminhadas.to_csv(
    caminho_portas_encaminhadas,
    index=False,
    date_format='%Y-%m-%d',
    float_format='%.15g',
)

metadata_integracao = {
    'formato_id_psd': '{ESTACAO}_P{PLATAFORMA}',
    'formato_id_porta_pipeline': (
        '{ESTACAO}_P{PLATAFORMA}_PDM{PORTA:02d}'
    ),
    'quantidade_plataformas_modelo_7': int(
        saida_modelo_7['id_psd'].nunique()
    ),
    'quantidade_plataformas_modelo_5': int(
        mapa_portas['id_psd'].nunique()
    ),
    'quantidade_portas_mapeadas': int(len(mapa_portas)),
    'plataformas_sem_mapeamento': plataformas_sem_mapeamento,
    'periodo_previsoes_modelo_7': {
        'inicio': str(saida_modelo_7['data_referencia'].min().date()),
        'fim': str(saida_modelo_7['data_referencia'].max().date()),
    },
    'periodo_features_modelo_5': {
        'inicio': str(inicio_painel_modelo_5.date()),
        'fim': str(fim_painel_modelo_5.date()),
        'regra_cobertura': 'presença da porta na plataforma e data exatas',
    },
    'quantidade_por_status': {
        chave: int(valor)
        for chave, valor in resumo_status['quantidade'].items()
    },
    'pares_plataforma_data_encaminhados': int(len(plataformas_elegiveis)),
    'linhas_porta_encaminhadas': int(len(portas_encaminhadas)),
    'arquivos': {
        'mapa_portas': caminho_mapa_portas.name,
        'saida_modelo_7': caminho_saida_modelo_7.name,
        'portas_encaminhadas': caminho_portas_encaminhadas.name,
    },
    'gerado_em_utc': datetime.now(timezone.utc).isoformat(),
}
caminho_metadata_integracao.write_text(
    json.dumps(metadata_integracao, indent=2, ensure_ascii=False) + '\n',
    encoding='utf-8',
)

print('Arquivos da Task 3 gerados:')
for caminho in (
    caminho_mapa_portas,
    caminho_saida_modelo_7,
    caminho_portas_encaminhadas,
    caminho_metadata_integracao,
):
    print('-', caminho)

# Task 4 — Executar o Modelo 5 e priorizar as portas

Nesta etapa, o Random Forest serializado é carregado e executado somente sobre as portas encaminhadas pelo Modelo 7. Cada porta recebe uma probabilidade, uma decisão pelo limiar do Modelo 5 e uma posição no ranking de risco da plataforma.

A saída operacional não utiliza o target. Como os dados são históricos, o target é mantido separadamente apenas para comparar o Modelo 5 isolado com o pipeline completo no período em que ambos possuem cobertura.

In [ ]:
caminho_portas_encaminhadas = (
    DIRETORIO_PLANILHAS / 'portas_encaminhadas_modelo_5.csv'
)
nome_modelo_5_inferencia = (
    'modelo_5_com_score_modelo_7.joblib'
    if USAR_MODELO_5_COM_SCORE_7 else 'modelo_5_random_forest.joblib'
)
nome_metadata_5_inferencia = (
    'metadata_modelo_5_com_score_modelo_7.json'
    if USAR_MODELO_5_COM_SCORE_7 else 'metadata_modelo_5.json'
)
caminho_modelo_5 = DIRETORIO_MODELOS_PIPELINE / nome_modelo_5_inferencia
caminho_metadata_5 = DIRETORIO_MODELOS_PIPELINE / nome_metadata_5_inferencia

with caminho_metadata_5.open(encoding='utf-8') as arquivo:
    metadata_modelo_5_inferencia = json.load(arquivo)

features_inferencia = metadata_modelo_5_inferencia['features']
limiar_modelo_5_inferencia = float(
    metadata_modelo_5_inferencia['limiar']
)
modelo_5_inferencia = joblib.load(caminho_modelo_5)
assert list(modelo_5_inferencia.feature_names_in_) == features_inferencia
features_painel_inferencia = [
    feature for feature in features_inferencia if feature != 'score_modelo_7'
]
assert set(features_painel_inferencia).issubset(painel.columns)
if USAR_MODELO_5_COM_SCORE_7:
    assert features_inferencia[-1] == 'score_modelo_7'
# Evita diferenças de arredondamento entre execuções paralelas.
modelo_5_inferencia.set_params(n_jobs=1)

portas_para_inferencia = pd.read_csv(
    caminho_portas_encaminhadas,
    parse_dates=['data_referencia'],
)
contexto_portas = portas_para_inferencia.drop(columns=['num_porta'])
features_historicas = painel[[
    'Data',
    'id_porta',
    *features_painel_inferencia,
]].rename(columns={
    'Data': 'data_referencia',
    'id_porta': 'id_porta_modelo_5',
})

portas_com_features = contexto_portas.merge(
    features_historicas,
    on=['data_referencia', 'id_porta_modelo_5'],
    how='left',
    validate='one_to_one',
    indicator=True,
)
assert portas_com_features['_merge'].eq('both').all()
portas_com_features = portas_com_features.drop(columns=['_merge'])
if USAR_MODELO_5_COM_SCORE_7:
    portas_com_features['score_modelo_7'] = (
        portas_com_features['probabilidade_plataforma']
    )
assert len(portas_com_features) == len(portas_para_inferencia)
assert not portas_com_features[features_inferencia].isna().any().any()
assert np.isfinite(portas_com_features[features_inferencia].to_numpy(dtype=float)).all()
portas_com_features = portas_com_features.sort_values(
    ['data_referencia', 'id_psd', 'id_porta_pipeline']
).reset_index(drop=True)

portas_com_features['probabilidade_porta'] = (
    modelo_5_inferencia.predict_proba(
        portas_com_features[features_inferencia]
    )[:, 1] if not portas_com_features.empty else np.array([], dtype=float)
)
portas_com_features['limiar_modelo_5'] = limiar_modelo_5_inferencia
portas_com_features['alerta_porta'] = (
    portas_com_features['probabilidade_porta']
    >= limiar_modelo_5_inferencia
).astype(int)
portas_com_features['posicao_risco_na_plataforma'] = (
    portas_com_features.groupby(
        ['data_referencia', 'id_psd']
    )['probabilidade_porta']
    .rank(method='first', ascending=False)
    .astype(int)
)
portas_com_features['porta_maior_risco_plataforma'] = (
    portas_com_features['posicao_risco_na_plataforma'].eq(1)
).astype(int)

colunas_saida_operacional = [
    'data_referencia',
    'id_psd',
    'id_porta_pipeline',
    'id_porta_modelo_5',
    'probabilidade_plataforma',
    'limiar_modelo_7',
    'probabilidade_porta',
    'limiar_modelo_5',
    'alerta_porta',
    'posicao_risco_na_plataforma',
    'porta_maior_risco_plataforma',
]
saida_portas_modelo_5 = portas_com_features[colunas_saida_operacional].copy()
portas_priorizadas = (
    saida_portas_modelo_5[saida_portas_modelo_5['alerta_porta'].eq(1)]
    .sort_values([
        'data_referencia',
        'probabilidade_plataforma',
        'id_psd',
        'posicao_risco_na_plataforma',
    ], ascending=[True, False, True, True])
    .reset_index(drop=True)
)

print(f'Registros de porta/dia avaliados pelo Modelo 5: {len(saida_portas_modelo_5):,}')
print(f'Alertas de porta/dia: {len(portas_priorizadas):,}')
display(portas_priorizadas.head())

In [ ]:
def calcular_metricas_binarias(y_real, predicao, score):
    """Métricas de amostras rotuladas; indefinidas são registradas como null."""
    y_real = np.asarray(y_real)
    predicao = np.asarray(predicao)
    score = np.asarray(score)
    tn, fp, fn, tp = confusion_matrix(
        y_real, predicao, labels=[0, 1]
    ).ravel() if len(y_real) else (0, 0, 0, 0)
    return {
        'precision': float(precision_score(y_real, predicao, zero_division=0))
            if len(y_real) else None,
        'recall': float(recall_score(y_real, predicao, zero_division=0))
            if len(y_real) else None,
        'f1': float(f1_score(y_real, predicao, zero_division=0))
            if len(y_real) else None,
        'pr_auc': float(average_precision_score(y_real, score))
            if (y_real == 1).any() else None,
        'roc_auc': float(roc_auc_score(y_real, score))
            if len(np.unique(y_real)) == 2 else None,
        'alertas': int(predicao.sum()),
        'tn': int(tn), 'fp': int(fp), 'fn': int(fn), 'tp': int(tp),
        'suporte_negativo': int((y_real == 0).sum()),
        'suporte_positivo': int((y_real == 1).sum()),
    }


def reducao_percentual(depois, antes):
    return float((1 - depois / antes) * 100) if antes else None


inicio_periodo_comum = max(
    painel['Data'].min(), saida_modelo_7['data_referencia'].min()
)
fim_periodo_comum = min(
    painel['Data'].max(), saida_modelo_7['data_referencia'].max()
)
metricas_portas_encaminhadas = None
metricas_modelo_5_periodo_comum = None
metricas_pipeline_periodo_comum = None
reducao_candidatas_percentual = None
reducao_alertas_percentual = None
avaliacao_comum = pd.DataFrame()
comparacao_metricas = pd.DataFrame()
status_avaliacao = 'desativada'

if AVALIAR_HISTORICO:
    if 'y' not in painel:
        status_avaliacao = 'sem_target'
    elif inicio_periodo_comum > fim_periodo_comum:
        status_avaliacao = 'sem_periodo_comum'
    else:
        mapa_plataformas = (
            mapa_portas[['id_plat_modelo_5', 'id_psd']]
            .drop_duplicates().set_index('id_plat_modelo_5')['id_psd']
        )
        avaliacao_comum = painel.loc[
            painel['Data'].between(inicio_periodo_comum, fim_periodo_comum)
            & painel['id_plat'].isin(mapa_plataformas.index)
        ].copy()
        avaliacao_comum['id_psd'] = avaliacao_comum['id_plat'].map(mapa_plataformas)
        avaliacao_comum = avaliacao_comum.merge(
            saida_modelo_7[[
                'data_referencia', 'id_psd',
                'probabilidade_plataforma', 'alerta_plataforma',
            ]],
            left_on=['Data', 'id_psd'],
            right_on=['data_referencia', 'id_psd'],
            how='inner',
            validate='many_to_one',
        )
        if avaliacao_comum.empty:
            status_avaliacao = 'sem_periodo_comum'
        elif avaliacao_comum['y'].isna().any():
            status_avaliacao = 'target_incompleto'
        elif not avaliacao_comum['y'].isin([0, 1]).all():
            raise ValueError('O target histórico y deve conter apenas 0 ou 1.')
        else:
            status_avaliacao = 'concluida'
            if USAR_MODELO_5_COM_SCORE_7:
                avaliacao_comum['score_modelo_7'] = (
                    avaliacao_comum['probabilidade_plataforma']
                )
            avaliacao_comum['probabilidade_porta'] = modelo_5_inferencia.predict_proba(
                avaliacao_comum[features_inferencia]
            )[:, 1]
            avaliacao_comum['alerta_modelo_5_isolado'] = (
                avaliacao_comum['probabilidade_porta'] >= limiar_modelo_5_inferencia
            ).astype(int)
            if USAR_MODELO_5_COM_SCORE_7:
                avaliacao_comum['alerta_pipeline'] = (
                    avaliacao_comum['alerta_modelo_5_isolado']
                )
                avaliacao_comum['score_pipeline'] = (
                    avaliacao_comum['probabilidade_porta']
                )
            else:
                avaliacao_comum['alerta_pipeline'] = (
                    avaliacao_comum['alerta_plataforma']
                    * avaliacao_comum['alerta_modelo_5_isolado']
                ).astype(int)
                avaliacao_comum['score_pipeline'] = np.where(
                    avaliacao_comum['alerta_plataforma'].eq(1),
                    avaliacao_comum['probabilidade_porta'], 0.0,
                )
            # O target é associado somente aqui, para avaliação.
            rotulos_encaminhados = portas_com_features.merge(
                avaliacao_comum[['Data', 'id_porta', 'y']],
                left_on=['data_referencia', 'id_porta_modelo_5'],
                right_on=['Data', 'id_porta'],
                how='left', validate='one_to_one',
            )
            if rotulos_encaminhados['y'].isna().any():
                raise ValueError('Há portas encaminhadas sem target no período avaliado.')
            metricas_portas_encaminhadas = calcular_metricas_binarias(
                rotulos_encaminhados['y'],
                rotulos_encaminhados['alerta_porta'],
                rotulos_encaminhados['probabilidade_porta'],
            )
            metricas_modelo_5_periodo_comum = calcular_metricas_binarias(
                avaliacao_comum['y'], avaliacao_comum['alerta_modelo_5_isolado'],
                avaliacao_comum['probabilidade_porta'],
            )
            metricas_pipeline_periodo_comum = calcular_metricas_binarias(
                avaliacao_comum['y'], avaliacao_comum['alerta_pipeline'],
                avaliacao_comum['score_pipeline'],
            )
            comparacao_metricas = pd.DataFrame([
                {'cenario': 'Modelo 5 isolado', **metricas_modelo_5_periodo_comum},
                {'cenario': 'Pipeline Modelo 7 → Modelo 5', **metricas_pipeline_periodo_comum},
            ])
            comparacao_exibicao = comparacao_metricas.copy()
            colunas_percentuais = ['precision', 'recall', 'f1', 'pr_auc', 'roc_auc']
            comparacao_exibicao[colunas_percentuais] *= 100
            display(comparacao_exibicao[[
                'cenario', *colunas_percentuais, 'alertas',
            ]].round(2))
            reducao_candidatas_percentual = reducao_percentual(
                len(portas_com_features), len(avaliacao_comum)
            )
            reducao_alertas_percentual = reducao_percentual(
                metricas_pipeline_periodo_comum['alertas'],
                metricas_modelo_5_periodo_comum['alertas'],
            )

print(f'Avaliação histórica: {status_avaliacao}')
if status_avaliacao == 'concluida':
    print(f'Redução de registros avaliados: {reducao_candidatas_percentual:.2f}%')
    print('Redução de alertas finais (%):', reducao_alertas_percentual)


In [ ]:
resumo_plataformas = (
    portas_com_features.sort_values(
        ['data_referencia', 'id_psd', 'probabilidade_porta'],
        ascending=[True, True, False],
    )
    .groupby(['data_referencia', 'id_psd'], as_index=False)
    .agg(
        probabilidade_plataforma=('probabilidade_plataforma', 'first'),
        porta_maior_risco=('id_porta_pipeline', 'first'),
        maior_probabilidade_porta=('probabilidade_porta', 'first'),
        quantidade_portas_priorizadas=('alerta_porta', 'sum'),
    )
)
resumo_plataformas['status_final'] = np.where(
    resumo_plataformas['quantidade_portas_priorizadas'].gt(0),
    'portas_priorizadas',
    'alerta_plataforma_sem_porta_acima_limiar',
)

caminho_saida_portas = (
    DIRETORIO_PLANILHAS / 'saida_portas_modelo_5.csv'
)
caminho_saida_final = (
    DIRETORIO_PLANILHAS / 'saida_final_portas_priorizadas.csv'
)
caminho_resumo_plataformas = (
    DIRETORIO_PLANILHAS / 'resumo_plataformas_pipeline.csv'
)
caminho_metricas_pipeline = (
    DIRETORIO_ARTEFATOS / 'metricas_pipeline_teste.json'
)

saida_portas_modelo_5.to_csv(
    caminho_saida_portas, index=False, date_format='%Y-%m-%d',
    float_format='%.15g',
)
portas_priorizadas.to_csv(
    caminho_saida_final, index=False, date_format='%Y-%m-%d',
    float_format='%.15g',
)
resumo_plataformas.to_csv(
    caminho_resumo_plataformas, index=False, date_format='%Y-%m-%d',
    float_format='%.15g',
)

resultado_task_4 = {
    'status_avaliacao': status_avaliacao,
    'periodo_comum': {
        'inicio': str(inicio_periodo_comum.date()) if inicio_periodo_comum <= fim_periodo_comum else None,
        'fim': str(fim_periodo_comum.date()) if inicio_periodo_comum <= fim_periodo_comum else None,
    },
    'target_avaliacao': (
        'y do Modelo 5: qualquer ZM01 da porta em (D, D+7]'
    ),
    'portas_no_periodo_comum': int(len(avaliacao_comum)),
    'portas_encaminhadas': int(len(portas_com_features)),
    'portas_priorizadas': int(len(portas_priorizadas)),
    'pares_plataforma_data_encaminhados': int(len(resumo_plataformas)),
    'pares_sem_porta_acima_limiar': int(
        resumo_plataformas['quantidade_portas_priorizadas'].eq(0).sum()
    ),
    'reducao_portas_avaliadas_percentual': reducao_candidatas_percentual,
    'reducao_alertas_percentual': reducao_alertas_percentual,
    'metricas_portas_encaminhadas': metricas_portas_encaminhadas,
    'metricas_modelo_5_isolado_periodo_comum': (
        metricas_modelo_5_periodo_comum
    ),
    'metricas_pipeline_periodo_comum': metricas_pipeline_periodo_comum,
    'arquivos': {
        'scores_portas_encaminhadas': caminho_saida_portas.name,
        'portas_priorizadas': caminho_saida_final.name,
        'resumo_plataformas': caminho_resumo_plataformas.name,
    },
    'gerado_em_utc': datetime.now(timezone.utc).isoformat(),
}
caminho_metricas_pipeline.write_text(
    json.dumps(resultado_task_4, indent=2, ensure_ascii=False, allow_nan=False) + '\n',
    encoding='utf-8',
)

print('Arquivos da Task 4 gerados:')
for caminho in (
    caminho_saida_portas,
    caminho_saida_final,
    caminho_resumo_plataformas,
    caminho_metricas_pipeline,
):
    print('-', caminho)

# Task 6 — Resultado da execução

O relatório final abaixo reúne todas as plataformas previstas. Quando uma plataforma não foi analisada pelo Modelo 5, os resultados de porta ficam vazios; isso não equivale a prever que não haverá falha.

A avaliação histórica, quando ativada, fica em arquivo separado. Nenhum target é exigido para gerar a lista priorizada.


In [ ]:
# Preserva todas as plataformas, incluindo alertas sem detalhamento por porta.
status_plataformas = saida_modelo_7.merge(
    resumo_plataformas.drop(columns=['probabilidade_plataforma', 'status_final']),
    on=['data_referencia', 'id_psd'],
    how='left',
    validate='one_to_one',
    indicator='_detalhamento',
)
analisadas = status_plataformas['_detalhamento'].eq('both')
encaminhadas = status_plataformas['status_encaminhamento'].str.startswith(
    'encaminhar_modelo_5'
)
if not analisadas.equals(encaminhadas):
    raise ValueError('Nem todas as plataformas encaminhadas possuem resultado do Modelo 5.')
status_plataformas['quantidade_portas_priorizadas'] = (
    status_plataformas['quantidade_portas_priorizadas'].astype('Int64')
)
status_plataformas['quantidade_portas_avaliadas'] = np.where(
    analisadas, status_plataformas['quantidade_portas_com_features_modelo_5'], 0
)
status_plataformas['status_final'] = status_plataformas['status_encaminhamento']
com_alerta_porta = (
    status_plataformas['quantidade_portas_priorizadas'].fillna(0).gt(0)
)
status_plataformas.loc[analisadas & ~com_alerta_porta, 'status_final'] = (
    'alerta_plataforma_sem_porta_acima_limiar'
)
status_plataformas.loc[analisadas & com_alerta_porta, 'status_final'] = (
    'portas_priorizadas'
)
parcial = status_plataformas['status_encaminhamento'].eq(
    'encaminhar_modelo_5_cobertura_parcial'
)
status_plataformas.loc[parcial & com_alerta_porta, 'status_final'] = (
    'portas_priorizadas_cobertura_parcial'
)
status_plataformas = status_plataformas.drop(columns='_detalhamento')
caminho_status_plataformas = DIRETORIO_PLANILHAS / 'status_plataformas_pipeline.csv'
status_plataformas.to_csv(
    caminho_status_plataformas, index=False, date_format='%Y-%m-%d',
    float_format='%.15g',
)

resumo_execucao = {
    'horizonte_dias': 7,
    'periodo_solicitado': {
        'inicio': str(INICIO_INFERENCIA_MODELO_7.date()),
        'fim': str(FIM_INFERENCIA_MODELO_7.date()),
    },
    'retreino_modelo_5': RETREINAR_MODELO_5,
    'avaliacao_historica_solicitada': AVALIAR_HISTORICO,
    'status_avaliacao': status_avaliacao,
    'modelos': {
        'modelo_7': {
            'arquivo': str(CAMINHO_CATBOOST.relative_to(RAIZ_REPOSITORIO)),
            'sha256': sha256(CAMINHO_CATBOOST.read_bytes()).hexdigest(),
            'limiar': float(metadata_modelo_7['limiar']),
        },
        'modelo_5': {
            'arquivo': str(caminho_modelo_5.relative_to(RAIZ_REPOSITORIO)),
            'arquivo_metadata': str(caminho_metadata_5.relative_to(RAIZ_REPOSITORIO)),
            'sha256': sha256(caminho_modelo_5.read_bytes()).hexdigest(),
            'sha256_metadata': sha256(caminho_metadata_5.read_bytes()).hexdigest(),
            'limiar': limiar_modelo_5_inferencia,
        },
    },
    'contagens': {
        'previsoes_plataforma_dia': len(status_plataformas),
        'alertas_plataforma_dia': int(status_plataformas['alerta_plataforma'].sum()),
        'plataformas_dia_analisadas_pelo_modelo_5': int(analisadas.sum()),
        'registros_porta_dia_avaliados': len(saida_portas_modelo_5),
        'alertas_porta_dia': len(portas_priorizadas),
        'portas_distintas_com_alerta': int(portas_priorizadas['id_porta_pipeline'].nunique()),
    },
    'quantidade_por_status_final': {
        str(k): int(v) for k, v in status_plataformas['status_final'].value_counts().items()
    },
    'arquivos': {
        'status_todas_plataformas': caminho_status_plataformas.name,
        'portas_avaliadas': caminho_saida_portas.name,
        'portas_priorizadas': caminho_saida_final.name,
        'avaliacao': caminho_metricas_pipeline.name,
    },
    'gerado_em_utc': datetime.now(timezone.utc).isoformat(),
}
caminho_resumo_execucao = DIRETORIO_ARTEFATOS / 'resumo_execucao_pipeline.json'
caminho_resumo_execucao.write_text(
    json.dumps(resumo_execucao, indent=2, ensure_ascii=False, allow_nan=False) + '\n',
    encoding='utf-8',
)
print('Execução concluída.')
display(pd.Series(resumo_execucao['contagens'], name='quantidade').to_frame())
print(f'Resultados em: {DIRETORIO_ARTEFATOS}')


# Task 7 — Gerar a lista operacional de manutenção

A saída completa mantém todas as portas que superaram o limiar do Random Forest para permitir auditoria e avaliação. Esta etapa cria uma segunda visão, mais direta, contendo somente as portas de maior risco de cada plataforma e data.

Com a configuração padrão, o pipeline recomenda uma porta por plataforma alertada. O período de sete dias da previsão também é explicitado em cada linha.


In [ ]:
if (
    isinstance(MAXIMO_PORTAS_RECOMENDADAS_POR_PLATAFORMA, bool)
    or not isinstance(MAXIMO_PORTAS_RECOMENDADAS_POR_PLATAFORMA, int)
    or MAXIMO_PORTAS_RECOMENDADAS_POR_PLATAFORMA < 1
):
    raise ValueError(
        'MAXIMO_PORTAS_RECOMENDADAS_POR_PLATAFORMA deve ser um inteiro maior que zero.'
    )

recomendacoes_manutencao = (
    portas_priorizadas.sort_values(
        [
            'data_referencia',
            'id_psd',
            'probabilidade_porta',
            'id_porta_pipeline',
        ],
        ascending=[True, True, False, True],
    )
    .groupby(['data_referencia', 'id_psd'], as_index=False, group_keys=False)
    .head(MAXIMO_PORTAS_RECOMENDADAS_POR_PLATAFORMA)
    .reset_index(drop=True)
    .copy()
)
recomendacoes_manutencao['inicio_horizonte_previsao'] = (
    recomendacoes_manutencao['data_referencia'] + pd.Timedelta(days=1)
)
recomendacoes_manutencao['fim_horizonte_previsao'] = (
    recomendacoes_manutencao['data_referencia'] + pd.Timedelta(days=7)
)
recomendacoes_manutencao['motivo_recomendacao'] = (
    'plataforma e porta acima dos respectivos limiares'
)
colunas_recomendacao = [
    'data_referencia',
    'inicio_horizonte_previsao',
    'fim_horizonte_previsao',
    'id_psd',
    'id_porta_pipeline',
    'id_porta_modelo_5',
    'probabilidade_plataforma',
    'probabilidade_porta',
    'posicao_risco_na_plataforma',
    'motivo_recomendacao',
]
recomendacoes_manutencao = recomendacoes_manutencao[colunas_recomendacao]

if recomendacoes_manutencao.duplicated(
    ['data_referencia', 'id_psd', 'id_porta_pipeline']
).any():
    raise ValueError('A lista operacional contém recomendações duplicadas.')
quantidade_maxima_observada = (
    recomendacoes_manutencao.groupby(['data_referencia', 'id_psd']).size().max()
    if not recomendacoes_manutencao.empty
    else 0
)
if quantidade_maxima_observada > MAXIMO_PORTAS_RECOMENDADAS_POR_PLATAFORMA:
    raise ValueError('O limite de recomendações por plataforma não foi respeitado.')

caminho_recomendacoes = DIRETORIO_PLANILHAS / 'recomendacoes_manutencao.csv'
recomendacoes_manutencao.to_csv(
    caminho_recomendacoes, index=False, date_format='%Y-%m-%d',
    float_format='%.15g',
)

metricas_recomendacoes = None
if status_avaliacao == 'concluida':
    chaves_recomendadas = recomendacoes_manutencao[[
        'data_referencia', 'id_porta_modelo_5'
    ]].assign(alerta_recomendacao=1)
    avaliacao_recomendacoes = avaliacao_comum.merge(
        chaves_recomendadas,
        left_on=['data_referencia', 'id_porta'],
        right_on=['data_referencia', 'id_porta_modelo_5'],
        how='left',
        validate='one_to_one',
    )
    avaliacao_recomendacoes['alerta_recomendacao'] = (
        avaliacao_recomendacoes['alerta_recomendacao'].fillna(0).astype(int)
    )
    metricas_recomendacoes_completas = calcular_metricas_binarias(
        avaliacao_recomendacoes['y'],
        avaliacao_recomendacoes['alerta_recomendacao'],
        avaliacao_recomendacoes['alerta_recomendacao'].astype(float),
    )
    metricas_recomendacoes = {
        chave: metricas_recomendacoes_completas[chave]
        for chave in (
            'precision', 'recall', 'f1', 'alertas', 'tn', 'fp', 'fn', 'tp',
            'suporte_negativo', 'suporte_positivo',
        )
    }

resultado_task_4['metricas_saida_operacional_periodo_comum'] = metricas_recomendacoes
resultado_task_4['arquivos']['recomendacoes_manutencao'] = caminho_recomendacoes.name
resultado_task_4['gerado_em_utc'] = datetime.now(timezone.utc).isoformat()
caminho_metricas_pipeline.write_text(
    json.dumps(resultado_task_4, indent=2, ensure_ascii=False, allow_nan=False) + '\n',
    encoding='utf-8',
)

resumo_execucao['maximo_portas_recomendadas_por_plataforma'] = (
    MAXIMO_PORTAS_RECOMENDADAS_POR_PLATAFORMA
)
resumo_execucao['contagens']['recomendacoes_porta_dia'] = len(recomendacoes_manutencao)
resumo_execucao['contagens']['portas_distintas_recomendadas'] = int(
    recomendacoes_manutencao['id_porta_pipeline'].nunique()
)
resumo_execucao['arquivos']['recomendacoes_manutencao'] = caminho_recomendacoes.name
resumo_execucao['gerado_em_utc'] = datetime.now(timezone.utc).isoformat()
caminho_resumo_execucao.write_text(
    json.dumps(resumo_execucao, indent=2, ensure_ascii=False, allow_nan=False) + '\n',
    encoding='utf-8',
)

print(f'Recomendações geradas: {len(recomendacoes_manutencao):,}')
print(f'Arquivo: {caminho_recomendacoes}')
if metricas_recomendacoes is not None:
    print(
        'Precisão da lista operacional: '
        f"{metricas_recomendacoes['precision'] * 100:.2f}% | "
        'Recall: '
        f"{metricas_recomendacoes['recall'] * 100:.2f}%"
    )
display(recomendacoes_manutencao.head(10))


# Task 8 — Execução automatizada

O arquivo `executar_pipeline.py` permite rodar este notebook por comando, informando período, pasta de saída, quantidade máxima de portas e uso da avaliação histórica. A execução ocorre sobre uma cópia em memória e não modifica o notebook original.

```bash
python models/pipeline-sequencial/executar_pipeline.py --inicio 2025-10-01 --fim 2026-03-21 --pasta-saida artefatos --max-portas 1
```

O comando força `RETREINAR_MODELO_5 = False`, portanto é destinado somente à inferência com os modelos já salvos.


# Task 9 — Validação e entrega

Após a execução, `validar_pipeline.py` verifica automaticamente as chaves, duplicidades, decisões pelos limiares, ranking, horizonte de sete dias, contagens, ausência de target nas saídas e hashes dos modelos.

```bash
python models/pipeline-sequencial/validar_pipeline.py
```

As dependências e o guia completo estão em `models/pipeline-sequencial/requirements.txt` e `models/pipeline-sequencial/README.md`.


# Task 10 — Retreinar o Modelo 5 com o score do Modelo 7

Este experimento substitui o filtro rígido por uma integração suave: todas as portas com features continuam sendo avaliadas, e a probabilidade da plataforma produzida pelo Modelo 7 entra como a 72ª feature do Random Forest.

Para medir somente o efeito da nova feature, são treinados dois modelos com os mesmos dados e hiperparâmetros: um controle com as 71 features originais e outro com `score_modelo_7`. Para os dois modelos, o limiar é escolhido somente na validação: entre os valores que atingem a meta de 75% de recall, vence o que apresenta maior precisão. Outubro de 2025 permanece separado para o teste final.

O novo artefato é salvo separadamente e não substitui o Random Forest original. O experimento pode ser desativado com `TREINAR_MODELO_5_COM_SCORE_7 = False`.


In [ ]:
if TREINAR_MODELO_5_COM_SCORE_7:
    # O score precisa existir para todo o histórico usado pelos dois Random Forest.
    inicio_scores_historicos = max(
        painel['Data'].min(), pd.Timestamp('2023-11-01')
    )
    fim_scores_historicos = min(
        painel['Data'].max(), pd.Timestamp('2025-10-31')
    )
    entrada_historica_modelo_7 = preparar_entrada_modelo_7(
        tabela_diaria_modelo_7,
        metadata_modelo_7,
        inicio_scores_historicos,
        fim_scores_historicos,
    )
    scores_historicos_modelo_7 = prever_plataformas_modelo_7(
        entrada_historica_modelo_7,
        modelo_7_inferencia,
        metadata_modelo_7,
    )[[
        'data_referencia', 'id_psd', 'probabilidade_falha_7d'
    ]].rename(columns={'probabilidade_falha_7d': 'score_modelo_7'})

    mapa_plataforma_experimento = (
        mapa_portas[['id_plat_modelo_5', 'id_psd']]
        .drop_duplicates()
        .rename(columns={'id_plat_modelo_5': 'id_plat'})
    )
    base_experimento = painel.merge(
        mapa_plataforma_experimento,
        on='id_plat',
        how='inner',
        validate='many_to_one',
    ).merge(
        scores_historicos_modelo_7,
        left_on=['Data', 'id_psd'],
        right_on=['data_referencia', 'id_psd'],
        how='inner',
        validate='many_to_one',
    )
    if base_experimento['score_modelo_7'].isna().any():
        raise ValueError('Há linhas do experimento sem score do Modelo 7.')

    fim_treino_experimento = pd.Timestamp('2025-04-23')
    inicio_validacao_experimento = pd.Timestamp('2025-05-01')
    fim_validacao_experimento = pd.Timestamp('2025-09-23')
    inicio_teste_experimento = pd.Timestamp('2025-10-01')
    fim_teste_experimento = pd.Timestamp('2025-10-31')

    treino_experimento = base_experimento.loc[
        base_experimento['Data'].le(fim_treino_experimento)
    ].copy()
    validacao_experimento = base_experimento.loc[
        base_experimento['Data'].between(
            inicio_validacao_experimento, fim_validacao_experimento
        )
    ].copy()
    treino_final_experimento = base_experimento.loc[
        base_experimento['Data'].le(fim_validacao_experimento)
    ].copy()
    teste_experimento = base_experimento.loc[
        base_experimento['Data'].between(
            inicio_teste_experimento, fim_teste_experimento
        )
    ].copy()

    for nome_base, base_periodo in {
        'treino': treino_experimento,
        'validacao': validacao_experimento,
        'treino_final': treino_final_experimento,
        'teste': teste_experimento,
    }.items():
        if base_periodo.empty:
            raise ValueError(f'O conjunto {nome_base} do experimento está vazio.')

    features_controle = list(features_modelo_5)
    features_com_score_7 = [*features_controle, 'score_modelo_7']
    parametros_experimento = {
        'n_estimators': 400,
        'max_depth': 14,
        'min_samples_leaf': 40,
        'class_weight': 'balanced',
        'n_jobs': -1,
        'random_state': 42,
    }

    def selecionar_limiar_por_meta_recall(y_real, probabilidades, meta_recall):
        precision_curva, recall_curva, limiares_curva = precision_recall_curve(
            y_real, probabilidades
        )
        candidatos = pd.DataFrame({
            'limiar': limiares_curva,
            'precision': precision_curva[:-1],
            'recall': recall_curva[:-1],
        })
        candidatos = candidatos.loc[candidatos['recall'].ge(meta_recall)]
        if candidatos.empty:
            raise ValueError('Nenhum limiar atende à meta de recall.')
        return float(
            candidatos.sort_values(
                ['precision', 'limiar'], ascending=[False, False]
            ).iloc[0]['limiar']
        )

    configuracoes_experimento = {
        'Modelo 5 controle': features_controle,
        'Modelo 5 + score do Modelo 7': features_com_score_7,
    }
    resultados_experimento = []
    modelos_finais_experimento = {}
    probabilidades_validacao_por_modelo = {}
    probabilidades_teste_experimento = {}

    for nome_modelo, features_experimento in configuracoes_experimento.items():
        modelo_validacao_experimento = RandomForestClassifier(
            **parametros_experimento
        )
        modelo_validacao_experimento.fit(
            treino_experimento[features_experimento],
            treino_experimento['y'],
        )
        probabilidades_validacao_experimento = (
            modelo_validacao_experimento.predict_proba(
                validacao_experimento[features_experimento]
            )[:, 1]
        )
        probabilidades_validacao_por_modelo[nome_modelo] = (
            probabilidades_validacao_experimento
        )

    nome_modelo_controle = 'Modelo 5 controle'
    nome_modelo_combinado = 'Modelo 5 + score do Modelo 7'
    limiares_experimento = {
        nome_modelo: selecionar_limiar_por_meta_recall(
            validacao_experimento['y'],
            probabilidades_validacao_por_modelo[nome_modelo],
            META_RECALL_EXPERIMENTO,
        )
        for nome_modelo in configuracoes_experimento
    }
    recalls_validacao_experimento = {
        nome_modelo: float(recall_score(
            validacao_experimento['y'],
            probabilidades_validacao_por_modelo[nome_modelo] >= limiar,
        ))
        for nome_modelo, limiar in limiares_experimento.items()
    }

    for nome_modelo, features_experimento in configuracoes_experimento.items():
        limiar_experimento = limiares_experimento[nome_modelo]
        modelo_final_experimento = RandomForestClassifier(
            **parametros_experimento
        )
        modelo_final_experimento.fit(
            treino_final_experimento[features_experimento],
            treino_final_experimento['y'],
        )
        modelo_final_experimento.set_params(n_jobs=1)
        probabilidades_teste = modelo_final_experimento.predict_proba(
            teste_experimento[features_experimento]
        )[:, 1]
        predicoes_teste = probabilidades_teste >= limiar_experimento
        metricas = calcular_metricas_binarias(
            teste_experimento['y'], predicoes_teste, probabilidades_teste
        )
        resultados_experimento.append({
            'cenario': nome_modelo,
            'quantidade_features': len(features_experimento),
            'limiar': limiar_experimento,
            **metricas,
        })
        modelos_finais_experimento[nome_modelo] = modelo_final_experimento
        limiares_experimento[nome_modelo] = limiar_experimento
        probabilidades_teste_experimento[nome_modelo] = probabilidades_teste

    comparacao_score_modelo_7 = pd.DataFrame(resultados_experimento)
    controle = comparacao_score_modelo_7.iloc[0]
    combinado = comparacao_score_modelo_7.iloc[1]
    ganhos_score_modelo_7 = {
        metrica: float((combinado[metrica] - controle[metrica]) * 100)
        for metrica in ('precision', 'recall', 'f1', 'pr_auc', 'roc_auc')
    }

    modelo_5_com_score_7 = modelos_finais_experimento[nome_modelo_combinado]
    limiar_modelo_5_com_score_7 = limiares_experimento[nome_modelo_combinado]
    caminho_modelo_com_score_7 = (
        DIRETORIO_ARTEFATOS / 'modelo_5_com_score_modelo_7.joblib'
    )
    caminho_comparacao_score_7 = (
        DIRETORIO_PLANILHAS / 'comparacao_modelo_5_score_modelo_7.csv'
    )
    caminho_avaliacao_score_7 = (
        DIRETORIO_PLANILHAS / 'avaliacao_teste_modelo_5_score_modelo_7.csv'
    )
    caminho_metadata_score_7 = (
        DIRETORIO_ARTEFATOS / 'metadata_modelo_5_com_score_modelo_7.json'
    )

    joblib.dump(modelo_5_com_score_7, caminho_modelo_com_score_7, compress=3)
    comparacao_score_modelo_7.to_csv(
        caminho_comparacao_score_7, index=False, float_format='%.15g'
    )
    avaliacao_score_7 = teste_experimento[[
        'Data', 'id_psd', 'id_porta', 'y', 'score_modelo_7'
    ]].copy()
    avaliacao_score_7['probabilidade_modelo_controle'] = (
        probabilidades_teste_experimento['Modelo 5 controle']
    )
    avaliacao_score_7['probabilidade_modelo_com_score_7'] = (
        probabilidades_teste_experimento[nome_modelo_combinado]
    )
    avaliacao_score_7['predicao_modelo_com_score_7'] = (
        avaliacao_score_7['probabilidade_modelo_com_score_7']
        .ge(limiar_modelo_5_com_score_7).astype(int)
    )
    avaliacao_score_7.to_csv(
        caminho_avaliacao_score_7, index=False, date_format='%Y-%m-%d',
        float_format='%.15g',
    )

    metadata_score_7 = {
        'modelo': nome_modelo_combinado,
        'objetivo': 'usar o risco da plataforma como feature da previsão por porta',
        'feature_adicionada': 'score_modelo_7',
        'quantidade_features': len(features_com_score_7),
        'features': features_com_score_7,
        'criterio_limiar': (
            'maior precisão entre os limiares que atingem a meta de recall na validação'
        ),
        'meta_recall_validacao': META_RECALL_EXPERIMENTO,
        'recalls_validacao': recalls_validacao_experimento,
        'limiar': limiar_modelo_5_com_score_7,
        'parametros': parametros_experimento,
        'periodos': {
            'treino_limiar_fim': str(fim_treino_experimento.date()),
            'validacao_inicio': str(inicio_validacao_experimento.date()),
            'validacao_fim': str(fim_validacao_experimento.date()),
            'treino_final_fim': str(fim_validacao_experimento.date()),
            'teste_inicio': str(inicio_teste_experimento.date()),
            'teste_fim': str(fim_teste_experimento.date()),
        },
        'metricas_teste_controle': resultados_experimento[0],
        'metricas_teste_com_score_7': resultados_experimento[1],
        'ganhos_pontos_percentuais': ganhos_score_modelo_7,
        'modelo_7_origem': str(CAMINHO_CATBOOST.relative_to(RAIZ_REPOSITORIO)),
        'sha256_modelo_7': sha256(CAMINHO_CATBOOST.read_bytes()).hexdigest(),
        'limitacao': (
            'O CatBoost usou abril a setembro de 2025 para seleção de configuração; '
            'o teste final de outubro permaneceu fora dessa seleção.'
        ),
        'gerado_em_utc': datetime.now(timezone.utc).isoformat(),
    }
    caminho_metadata_score_7.write_text(
        json.dumps(metadata_score_7, indent=2, ensure_ascii=False, allow_nan=False) + '\n',
        encoding='utf-8',
    )

    comparacao_exibicao_score_7 = comparacao_score_modelo_7.copy()
    for coluna in ('precision', 'recall', 'f1', 'pr_auc', 'roc_auc'):
        comparacao_exibicao_score_7[coluna] *= 100
    display(comparacao_exibicao_score_7[[
        'cenario', 'quantidade_features', 'limiar', 'precision', 'recall',
        'f1', 'pr_auc', 'roc_auc', 'alertas',
    ]].round(2))
    print('Ganho do score do Modelo 7 em pontos percentuais:')
    display(pd.Series(ganhos_score_modelo_7, name='ganho_pp').round(2).to_frame())
    print(f'Novo modelo salvo separadamente em: {caminho_modelo_com_score_7}')
else:
    print('Experimento com score do Modelo 7 desativado.')


## Resultado do experimento

Com a meta de **75% de recall na validação**, o Modelo 5 com as 71 features originais obteve **9,88% de precisão**, **79,35% de recall** e **17,58% de F1** no teste. Ao adicionar o `score_modelo_7` como a 72ª feature, os resultados ficaram em **9,88% de precisão**, **78,67% de recall** e **17,56% de F1**. O modelo combinado detectou **697 das 886 falhas** e gerou 7.053 alertas.

A política de limiar elevou bastante o recall do modelo combinado: de 49,44% no experimento anterior para 78,67%. Entretanto, quando controle e modelo combinado são avaliados com a mesma meta, o `score_modelo_7` não melhora recall, precisão ou F1; ele reduz somente 59 alertas. Em relação ao pipeline anterior com filtro rígido, o recall é maior, mas a precisão e o F1 são menores. Por isso, a escolha operacional depende da prioridade dada a não perder falhas e da capacidade de atender ao volume de alertas.